# Binance candle backfill and inspection

This notebook uses the same backfill function as the command-line collector. It is safe to rerun because existing candles are upserted, not duplicated. It never places trades.

In [ ]:
from pathlib import Path
import pandas as pd
from crypto_trader.data.binance import backfill_historical_candles, open_candle_database

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "data" / "database" / "trading.db"
symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "DOGEUSDT"]
backfill_days = 30
connection = open_candle_database(database_path)

## Backfill

Change `symbols` or `backfill_days` above, then run this cell. A 30-day one-minute backfill is 43,200 rows per symbol and may take several minutes.

In [ ]:
processed_rows = backfill_historical_candles(
    connection,
    symbols=symbols,
    minutes=backfill_days * 24 * 60,
)
print(f"Processed {processed_rows:,} candle rows.")

## Inspect coverage and schema

The following cells show how many rows exist, the time range for each symbol, recent prices, and the actual SQLite schema.

In [ ]:
coverage = pd.read_sql_query(
    """
    SELECT exchange, symbol, interval_seconds, COUNT(*) AS rows,
           MIN(open_time_ms) AS first_open_time_ms,
           MAX(open_time_ms) AS last_open_time_ms
    FROM candles
    GROUP BY exchange, symbol, interval_seconds
    ORDER BY symbol
    """,
    connection,
)
coverage

In [ ]:
latest = pd.read_sql_query(
    """
    SELECT symbol, datetime(open_time_ms / 1000, 'unixepoch') AS open_time_utc,
           open, high, low, close, volume
    FROM candles
    WHERE interval_seconds = 60
    ORDER BY open_time_ms DESC
    LIMIT 25
    """,
    connection,
)
latest

In [ ]:
schema = pd.read_sql_query(
    "SELECT type, name, sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type, name",
    connection,
)
schema

In [ ]:
# Close the connection when finished so other services can use the database.
connection.close()